# Guardian Football — Player Annotation

Use Azure OpenAI to identify and highlight football player names mentioned in the latest Guardian football articles.

In [ ]:
## Cell 1 — Load the latest Guardian football articles (with headlines) from saved .pkl file

import pickle

with open("guardian_football_articles.pkl", "rb") as f:
    articles = pickle.load(f)

print(f"Loaded {len(articles)} articles.")
for i, a in enumerate(articles[:2]):
    headline = a["headline"] if isinstance(a, dict) else a[:80]
    print(f"  {i+1}. {headline}")


Loaded 50 articles.
  1. Nico O’Reilly’s fearless quality exposes collapsing Arsenal’s title credentials
  2. Guardiola calls for focus in title run-in after Haaland cuts down Arsenal
  3. Manchester City 2-1 Arsenal: Premier League – as it happened
  4. Manchester City 2-1 Arsenal: Premier League player ratings
  5. Erling Haaland sinks Arsenal to give Manchester City edge in title race


In [ ]:
## Cell 2 — Extract players and teams with Azure OpenAI (combined)

N_ARTICLES = 2   # ← change this to annotate more articles

import json
from urllib.parse import urlparse
from dotenv import find_dotenv
from openai import AzureOpenAI

# ── parse .env ───────────────────────────────────────────────────────────────
def read_env(path):
    env = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            env[key.strip()] = val.strip().strip('"').strip("'")
    return env

env = read_env(find_dotenv())

raw_base       = env.get("GPT_BASE", "")
parsed         = urlparse(raw_base)
azure_endpoint = f"{parsed.scheme}://{parsed.netloc}"
api_key        = env.get("GPT_KEY", "")
api_version    = env.get("GPT_VERSION", "2025-04-01-preview")
CHAT_MODEL     = env.get("GPT_CHAT_MODEL", "gpt-4o-mini-context-course-2026-afthibara")

client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=api_key,
    api_version=api_version,
)

ANNOTATION_PROMPT = """\
You are a football analyst. Given the text of a football article, extract:
1. Every football player name mentioned.
2. Every football club or national team name mentioned.

Return ONLY valid JSON in this exact format (no extra text):
{
  "players": ["Player One", "Player Two"],
  "teams":   ["Club A", "National Team B"]
}
Return empty arrays if nothing is found.\
"""

def annotate_article(text: str) -> dict:
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": ANNOTATION_PROMPT},
            {"role": "user",   "content": text[:4000]},
        ],
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

# Annotate the first N_ARTICLES articles (one API call each)
annotations = []
for i, article in enumerate(articles[:N_ARTICLES]):
    headline = article["headline"]
    print(f"Annotating {i+1}: {headline}")
    tags = annotate_article(article["body"])
    annotations.append(tags)
    print(f"   Players : {', '.join(tags['players']) or '(none)'}")
    print(f"   Teams   : {', '.join(tags['teams'])   or '(none)'}\n")


Annotating 1: Nico O’Reilly’s fearless quality exposes collapsing Arsenal’s title credentials
   Players : Erling Haaland, Gabriel Magalhães, Nico O’Reilly, Jérémy Doku, Paolo Maldini, Mikel Arteta
   Teams   : Manchester City, Arsenal, England

Annotating 2: Guardiola calls for focus in title run-in after Haaland cuts down Arsenal
   Players : Rayan Cherki, Kai Havertz, Gianluigi Donnarumma, Erling Haaland, Rodri, Gabriel Magalhães, Bernardo Silva, Viktor Gyökeres, Fabio Cannavaro
   Teams   : Manchester City, Arsenal, Burnley, Everton, Brentford, Bournemouth, Crystal Palace, Aston Villa, Newcastle United, Fulham, West Ham United

Annotating 3: Manchester City 2-1 Arsenal: Premier League – as it happened
   Players : Erling Haaland, Bernardo Silva, Havertz, Gabriel, Cannavaro, Gyokeres
   Teams   : Manchester City, Arsenal, Ipswich, Burnley, Everton, Brentford, Bournemouth, Crystal Palace, Aston Villa, Newcastle, Fulham, West Ham

Annotating 4: Manchester City 2-1 Arsenal: Premier Lea

In [ ]:
## Cell 3 — Display each article with players (yellow) and teams (green) highlighted

import re
from IPython.display import display, HTML

PLAYER_STYLE = "background-color:#ffe066; font-weight:bold; border-radius:3px; padding:0 2px;"
TEAM_STYLE   = "background-color:#b6f5b0; font-weight:bold; border-radius:3px; padding:0 2px;"

def highlight_entities(text: str, players: list[str], teams: list[str]) -> str:
    text = (text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
    # Highlight teams first, then players (sort longest first to avoid partial matches)
    for team in sorted(teams, key=len, reverse=True):
        pattern = re.compile(re.escape(team), re.IGNORECASE)
        text = pattern.sub(f'<span style="{TEAM_STYLE}">{team}</span>', text)
    for player in sorted(players, key=len, reverse=True):
        pattern = re.compile(re.escape(player), re.IGNORECASE)
        text = pattern.sub(f'<span style="{PLAYER_STYLE}">{player}</span>', text)
    return text

def make_badges(items: list[str], colour: str) -> str:
    return " ".join(
        f'<span style="background:{colour};color:white;border-radius:4px;'
        f'padding:1px 6px;margin:2px;font-size:0.85em;">{item}</span>'
        for item in items
    )

for i, (article, tags) in enumerate(zip(articles[:N_ARTICLES], annotations)):
    headline = article["headline"]
    body     = article["body"]
    players  = tags["players"]
    teams    = tags["teams"]

    highlighted = highlight_entities(body, players, teams)

    html = f"""
    <div style="border:1px solid #ccc;border-radius:6px;padding:16px;margin-bottom:24px;font-family:Georgia,serif;line-height:1.7;">
      <h3 style="margin-top:0;color:#1a1a1a;">{i+1}. {headline}</h3>
      <div style="margin-bottom:4px;">
        <strong>Players</strong> (yellow): {make_badges(players, '#c9a000') or '<em>none</em>'}
      </div>
      <div style="margin-bottom:8px;">
        <strong>Teams</strong> (green): {make_badges(teams, '#3a8c35') or '<em>none</em>'}
      </div>
      <hr style="border:none;border-top:1px solid #eee;margin:8px 0;">
      <p style="font-size:0.95em;color:#333;">{highlighted[:2000]}{'…' if len(body) > 2000 else ''}</p>
    </div>
    """
    display(HTML(html))


In [ ]:
## Cell 4 — Build paragraph-level tag index and save enriched data

import pickle
import re

def tag_paragraphs(body: str, players: list[str], teams: list[str]) -> list[dict]:
    """Split article into paragraphs and tag each with any players/teams it mentions."""
    paragraphs = [p.strip() for p in body.split("\n") if p.strip()]
    tagged = []
    for para in paragraphs:
        para_lower = para.lower()
        para_players = [p for p in players if p.lower() in para_lower]
        para_teams   = [t for t in teams   if t.lower() in para_lower]
        tagged.append({
            "text":    para,
            "players": para_players,
            "teams":   para_teams,
        })
    return tagged

# Build enriched articles list
enriched_articles = []
for article, tags in zip(articles[:N_ARTICLES], annotations):
    enriched = {
        "headline":   article["headline"],
        "body":       article["body"],
        "players":    tags["players"],
        "teams":      tags["teams"],
        "paragraphs": tag_paragraphs(article["body"], tags["players"], tags["teams"]),
    }
    enriched_articles.append(enriched)

# Save to disk
output_path = "guardian_football_annotated.pkl"
with open(output_path, "wb") as f:
    pickle.dump(enriched_articles, f)
print(f"Saved {len(enriched_articles)} annotated articles to '{output_path}'")

# Quick lookup demo — find all paragraphs mentioning a given player or team
def lookup_paragraphs(enriched: list[dict], tag: str) -> list[dict]:
    """Return all paragraphs across articles that mention the given player or team."""
    results = []
    for art in enriched:
        for para in art["paragraphs"]:
            if tag in para["players"] or tag in para["teams"]:
                results.append({"headline": art["headline"], "paragraph": para["text"]})
    return results

# Example: find paragraphs for the first player found across all articles
all_players = [p for art in enriched_articles for p in art["players"]]
if all_players:
    example_tag = all_players[0]
    hits = lookup_paragraphs(enriched_articles, example_tag)
    print(f"\nParagraphs mentioning '{example_tag}':")
    for h in hits[:3]:
        print(f"  [{h['headline']}]\n  {h['paragraph'][:120]}…\n")


Saved 50 annotated articles to 'guardian_football_annotated_all.pkl'

Paragraphs mentioning 'Erling Haaland':
  [Nico O’Reilly’s fearless quality exposes collapsing Arsenal’s title credentials]
  It’s not over, not over, not over yet. Although, let’s be honest, it kind of is over. Isn’t it, don’t you think, at the …

  [Guardiola calls for focus in title run-in after Haaland cuts down Arsenal]
  Pep Guardiola urged Manchester City not “to lose focus” after Sunday’s 2-1 win over Arsenal at the Etihad Stadium struck…

  [Manchester City 2-1 Arsenal: Premier League – as it happened]
  Match report: Man City 2-1 Arsenal The margins were always likely to be tight. In the event, with so much on the line, n…

